In [ ]:
#Name: Anisa Sachdev
#Section: G




: 

In [ ]:
#tasks
#task1

In [ ]:
import pandas as pd

df = pd.read_csv("clean_churn.csv")
df.isnull().sum()

In [ ]:
y = df['Churn'].map({'Yes':1, 'No':0})
x = df.drop(columns=['Churn'])
x

In [ ]:
cat_col = x.select_dtypes(include='object').columns
x = pd.get_dummies(x,cat_col,dtype=int)

x.info()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42, stratify=y)
model = DecisionTreeClassifier(max_depth=5, random_state=42)

model.fit(x_train,y_train)

y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

print("Tree Depth:", model.get_depth())

dt_accuracy = accuracy_score(y_test, y_test_pred)
dt_precision = precision_score(y_test, y_test_pred)
dt_recall = recall_score(y_test, y_test_pred)
dt_f1 = f1_score(y_test, y_test_pred)

print("Accuracy:", dt_accuracy)
print("precision:", dt_precision)
print("recall:", dt_recall)
print("f1:", dt_f1)





In [ ]:
# part 2

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

rf_model.fit(x_train,y_train)

y_test_pred_rf = rf_model.predict(x_test)

accuracy_rf = accuracy_score(y_test,y_test_pred_rf)
precision_rf = precision_score(y_test,y_test_pred_rf)
recall_rf = recall_score(y_test,y_test_pred_rf)
f1_rf = f1_score(y_test,y_test_pred_rf)


print("Accuracy:", accuracy_score(y_test, y_test_pred_rf))
print("precision:", precision_score(y_test, y_test_pred_rf))
print("recall:", recall_score(y_test, y_test_pred_rf))
print("f1:", f1_score(y_test, y_test_pred_rf))

In [ ]:
# Although random forest gives better results but in this particular example we are seeing that the results are not satisfying as 
# compare to single decision tree. 
# There can be multiple reasons for that: We have set the number of trees in random forest to be 200. As we have seen that decision tree
# does not perform well on high depth. Therefore, in my opinion this can be one reason for that. The other reasons are that the random 
# forest helps us to reduce overfitting but does not guarantee the optimal solution. 

In [ ]:
# part 3

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_rf = confusion_matrix(y_test, y_test_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp.plot()

plt.title("Random Forest Confusion Matrix")
plt.show()


In [ ]:
# task 6

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 score'],

    'Decision Tree': [
        accuracy_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred),
        recall_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred)
    ],

    'Random Forest': [
        accuracy_rf,
        precision_rf,
        recall_rf,
        f1_rf
    ]
})

comparison

In [ ]:
# we can see that the Decision has performed better overall then random forest. 
# some matrix has marginal difference and some have large difference
# recall is the most important due to imbalance

In [ ]:
# part 4

In [ ]:
from sklearn.model_selection import cross_val_score

dt_scores = cross_val_score(model, x, y, cv=5, scoring='f1')

rf_scores = cross_val_score(rf_model, x, y, cv=5, scoring='f1')

print("Decision Tree F1 score:", dt_scores)
print("Decision Tree Mean F1 score:", dt_scores.mean())
print("Decision Tree Std F1 score:", dt_scores.std())

print()

print("Random Forest F1 score:", rf_scores)
print("Random Forest Mean F1 score:", rf_scores.mean())
print("Random Forest Std F1 score:", rf_scores.std())


In [ ]:
#task 9
# The cross validation results are consistent with single split results from part 3.
# The Decision Tree has a higher mean F1-score so it performs better on average then random forest.
# These results from cross validation are a strong evidence that the results are consistent and 
# Decision tree has performed well as compare to random forest.

In [ ]:
# part 5

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring='f1'
)

grid_search.fit(x,y)


In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best cross-validated F1:", grid_search.best_score_)

In [ ]:
tuned_rf = grid_search.best_estimator_

y_pred_tuned = tuned_rf.predict(x_test)

accuracy = accuracy_score(y_test, y_pred_tuned)
precision = precision_score(y_test, y_pred_tuned)
recall = recall_score(y_test, y_pred_tuned)
f1 = f1_score(y_test, y_pred_tuned) 

print('Accuracy:', accuracy)
print('precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)

In [ ]:
# part 6:

In [ ]:
comparison = pd.DataFrame({
    "Model": ['Decision Tree', "Default Random Forest", "Tuned Random Forest"],
    "Accuracy": [dt_accuracy, accuracy_rf, accuracy],
    "Precision": [dt_precision, precision_rf, precision],
    "Recall": [dt_recall, recall_rf, recall],
    "F1-score": [dt_f1, f1_rf, f1]
})

comparison

In [ ]:
# I will choose tuned Random Forest then any other model.
# The reason is visible above in a table where we can see that tuned random forest model has achieved
# the better results than any other model.

In [ ]:
important = tuned_rf.feature_importances_

feature_importance = pd.Series(
    important, index = x.columns
).sort_values(ascending=False)

top_five = feature_importance.head(5)
print(top_five)

In [ ]:
top_five.plot(kind="barh")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Tuned Random Forest Feature Importance")
plt.show()

In [ ]:
# The 5 most important features are tenure, total_Charges, contract_month_to_month, and 
# monthly_charges and online_security_no.
# These results mostly agree with our lab2 where we found that features such as tenure, charges are
# important to find the customer churn.


In [ ]:
# part 7

In [ ]:
original_df = pd.read_csv("original_churn.csv")

original_df["customerID"].head()

In [ ]:
customer_ids = original_df.loc[x_test.index, "customerID"]

submission = pd.DataFrame({
    "customerID" : customer_ids,
    "Churn": y_pred_tuned
})

submission.to_csv("submission.csv", index=False)

submission.head()

In [ ]:
# task 17

In [ ]:
# I selected the Tuned Random forest model because it gave the optimal solution of all models.
# This model provided better results than lab3 simple Decisiontree model.
# The most important features were tenure, totalcharges, onlinesecurity etc
# The model has some limitations such as low recall which means the model failed to recognize some of the customers who churn.

In [ ]:
# task 18

In [ ]:
# If i had more time i would probably try diff models and see their results.
# I would also pedantically tune the existing features to see if the results are improved.